# SAGAN and BigGAN: Attention and Conditional Scaling

## Theory, mathematics, and implementation context

Convolutional networks build local features, but distant regions interact only after many layers. Self-attention makes a direct global interaction. For features $X\in\mathbb R^{N\times C}$, SAGAN computes $Q=XW_q$, $K=XW_k$, $V=XW_v$, then $A=\mathrm{softmax}(QK^T/\sqrt d)$ and $Y=X+\gamma AV$. Initializing learnable $\gamma=0$ starts as an identity path. Attention costs $O(N^2)$ in spatial positions, so it is typically placed at modest resolutions.

BigGAN scaled conditional synthesis with class-conditioned normalization, shared class embeddings, spectral normalization, large batches, and a projection discriminator. The discriminator score adds image/class compatibility $f(x)^Te_y$ to its unconditional score. Truncation controls the fidelity-diversity trade-off. This notebook uses compact CIFAR-10 models to teach SAGAN attention and BigGAN conditioning; it does not reproduce original 128–512px training scale.

## Case study setup

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
torch.manual_seed(7)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf=transforms.Compose([transforms.Resize(32),transforms.ToTensor(),transforms.Normalize((.5,)*3,(.5,)*3)])
ds=datasets.CIFAR10("./data",train=True,download=True,transform=tf)
loader=DataLoader(Subset(ds,range(4096)),batch_size=64,shuffle=True,num_workers=0,drop_last=True)
print(DEVICE,len(loader.dataset))

## SAGAN self-attention block

Keys are channel-reduced for efficiency. Query/source dimensions and the softmax axis are explicit.

In [ ]:
class SelfAttention2d(nn.Module):
    def __init__(self,c):
        super().__init__(); h=max(8,c//8); self.q=nn.Conv2d(c,h,1); self.k=nn.Conv2d(c,h,1); self.v=nn.Conv2d(c,c,1); self.gamma=nn.Parameter(torch.zeros(()))
    def forward(self,x):
        b,c,h,w=x.shape; q=self.q(x).flatten(2).transpose(1,2); k=self.k(x).flatten(2); v=self.v(x).flatten(2).transpose(1,2)
        a=torch.softmax(torch.bmm(q,k)*q.shape[-1]**-.5,dim=-1)
        return x+self.gamma*torch.bmm(a,v).transpose(1,2).reshape(b,c,h,w)
print(SelfAttention2d(64)(torch.randn(2,64,8,8)).shape)

## Conditional BatchNorm and generator blocks

Class embeddings predict channel scale and shift. BigGAN uses richer shared embeddings and hierarchical latent injection; this version illustrates the conditioning principle.

In [ ]:
class CBN(nn.Module):
    def __init__(self,c,nclass=10,edim=128):
        super().__init__(); self.bn=nn.BatchNorm2d(c,affine=False); self.emb=nn.Embedding(nclass,edim); self.scale=nn.Linear(edim,c); self.shift=nn.Linear(edim,c)
        nn.init.zeros_(self.scale.weight); nn.init.ones_(self.scale.bias); nn.init.zeros_(self.shift.weight); nn.init.zeros_(self.shift.bias)
    def forward(self,x,y): return self.bn(x)*self.scale(self.emb(y))[:,:,None,None]+self.shift(self.emb(y))[:,:,None,None]
class CBlock(nn.Module):
    def __init__(self,ci,co):
        super().__init__(); self.n1=CBN(ci); self.n2=CBN(co); self.c1=nn.Conv2d(ci,co,3,padding=1); self.c2=nn.Conv2d(co,co,3,padding=1); self.skip=nn.Conv2d(ci,co,1)
    def forward(self,x,y):
        u=F.interpolate(x,scale_factor=2,mode='nearest'); h=self.c1(F.relu(self.n1(u,y))); h=self.c2(F.relu(self.n2(h,y))); return h+self.skip(u)
class SAGANGenerator(nn.Module):
    def __init__(self):
        super().__init__(); self.fc=nn.Linear(128,256*4*4); self.blocks=nn.ModuleList([CBlock(256,128),CBlock(128,64),CBlock(64,32)]); self.attn=SelfAttention2d(64); self.rgb=nn.Conv2d(32,3,3,padding=1)
    def forward(self,z,y):
        x=self.fc(z).view(-1,256,4,4)
        for i,block in enumerate(self.blocks):
            x=block(x,y)
            if i==1: x=self.attn(x)
        return torch.tanh(self.rgb(F.relu(x)))
G=SAGANGenerator(); print(G(torch.randn(2,128),torch.tensor([0,1])).shape)

## BigGAN projection discriminator and spectral normalization

Spectral normalization controls the operator norm of discriminator layers. Projection conditioning adds compatibility between pooled image features and a class embedding.

In [ ]:
from torch.nn.utils import spectral_norm
class ProjectionD(nn.Module):
    def __init__(self):
        super().__init__(); self.features=nn.Sequential(spectral_norm(nn.Conv2d(3,64,4,2,1)),nn.LeakyReLU(.2),spectral_norm(nn.Conv2d(64,128,4,2,1)),nn.LeakyReLU(.2),SelfAttention2d(128),spectral_norm(nn.Conv2d(128,256,4,2,1)),nn.LeakyReLU(.2))
        self.score=spectral_norm(nn.Linear(256,1)); self.embed=nn.Embedding(10,256)
    def forward(self,x,y):
        h=self.features(x).sum((2,3)); return self.score(h)+(self.embed(y)*h).sum(1,keepdim=True)
D=ProjectionD(); print(D(torch.randn(2,3,32,32),torch.tensor([1,2])).shape)

## Hinge-loss training step

BigGAN-style hinge objective: $L_D=E\max(0,1-D(x,y))+E\max(0,1+D(G(z,y),y))$ and $L_G=-E D(G(z,y),y)$. One update is a mechanics demonstration, not a stable recipe.

In [ ]:
G=SAGANGenerator().to(DEVICE); D=ProjectionD().to(DEVICE)
oG=torch.optim.Adam(G.parameters(),2e-4,betas=(0,.999)); oD=torch.optim.Adam(D.parameters(),2e-4,betas=(0,.999))
x,y=next(iter(loader)); x=x.to(DEVICE); y=y.to(DEVICE); b=len(x)
fake=G(torch.randn(b,128,device=DEVICE),y).detach(); ld=F.relu(1-D(x,y)).mean()+F.relu(1+D(fake,y)).mean()
oD.zero_grad(); ld.backward(); oD.step()
lg=-D(G(torch.randn(b,128,device=DEVICE),y),y).mean(); oG.zero_grad(); lg.backward(); oG.step(); print(float(ld),float(lg))

## Truncation, attention cost, and scale

BigGAN truncation samples from a constrained latent distribution; radial clipping below is only an approximation. Holding z fixed and changing labels tests class control. At 32×32, full attention scores 1,024² positions; at 128×128, over 268 million scores per example. Truncation improves typicality at the cost of diversity.

In [ ]:
def truncate_z(z,threshold=.8):
    norm=z.norm(dim=1,keepdim=True).clamp_min(1e-8); return z*torch.clamp(threshold/norm,max=1.)
G.eval()
with torch.no_grad():
    z=truncate_z(torch.randn(10,128,device=DEVICE)); labels=torch.arange(10,device=DEVICE); imgs=G(z,labels).cpu()
plt.figure(figsize=(10,2)); plt.axis('off'); plt.imshow(utils.make_grid(imgs,nrow=10,normalize=True).permute(1,2,0));

## Key Takeaways

- SAGAN uses global attention; zero initialization lets its residual path start as identity.
- BigGAN conditions G with class normalization and D with projection scores.
- Spectral normalization and hinge loss are common stability tools.
- Truncation trades variety for typicality.
- Attention has quadratic cost in spatial positions.
- BigGAN quality relied on substantial data and training scale.